# AValidate · Vision Baseline (Local CPU / Integrated Graphics)

This notebook is configured specifically for **local machine execution** using your CPU (multi-threaded) or Integrated Graphics (Intel Arc 140V / AMD iGPU).

### Key Local Optimizations:
- **No Google Drive dependency**: Uses local repo paths (`_local/work` for scratch frames and `_local/drive` for checkpoints).
- **CPU / iGPU-friendly architecture**: Defaults to **MobileNet-V3 Small** or lightweight **VisionCNN** (orders of magnitude faster than ResNet on CPU).
- **Intel Arc & XPU Acceleration Ready**: Automatically detects `torch.xpu` if PyTorch Intel GPU support is enabled.
- **Windows-optimized data loading**: Uses `workers=0` to eliminate Windows multiprocessing process-spawning overhead.
- **W&B tracking enabled**: Authenticates via `.env` without exposing secrets in your git commits.
- **Resume capability**: All steps skip existing files so you can stop and restart anytime.

## 0 · Environment & Hardware Check

In [ ]:
import os, sys
import torch

# Force unbuffered streaming output for all cell subprocesses
os.environ["PYTHONUNBUFFERED"] = "1"

# Ensure notebook runs from repository root
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Working directory: {REPO_ROOT}")
print(f"PyTorch version:   {torch.__version__}")

# Detect hardware capabilities
has_cuda = torch.cuda.is_available()
has_xpu = hasattr(torch, 'xpu') and torch.xpu.is_available()
has_mps = getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()

try:
    import torch_directml
    dml_device = torch_directml.device()
    has_dml = True
except ImportError:
    has_dml = False
    dml_device = None

if has_cuda:
    device_desc = f"NVIDIA CUDA GPU ({torch.cuda.get_device_name(0)})"
elif has_xpu:
    device_desc = f"Intel Arc XPU GPU: {torch.xpu.get_device_name(0)}"
elif has_dml:
    device_desc = f"DirectML Integrated Graphics ({dml_device})"
elif has_mps:
    device_desc = "Apple MPS"
else:
    num_threads = torch.get_num_threads()
    device_desc = f"Multi-threaded CPU ({num_threads} threads)"

print(f"Active compute:    {device_desc}")

## 1 · Weights & Biases Experiment Tracking (Optional)
Reads your API key safely from `.env` (which is kept out of git) and authenticates to `GC2-AValidate/avalidate`.

In [ ]:
import os
from pathlib import Path

# Load credentials from local .env file
env_file = Path(REPO_ROOT) / '.env'
if env_file.exists():
    for line in env_file.read_text(encoding='utf-8').splitlines():
        if '=' in line and not line.strip().startswith('#'):
            k, v = line.strip().split('=', 1)
            os.environ[k.strip()] = v.strip()

WANDB_ENTITY = os.environ.get('WANDB_ENTITY', 'GC2-AValidate')
WANDB_PROJECT = os.environ.get('WANDB_PROJECT', 'avalidate')
api_key = os.environ.get('WANDB_API_KEY', '')

try:
    import wandb
    if api_key:
        os.environ['WANDB_ENTITY'] = WANDB_ENTITY
        os.environ['WANDB_PROJECT'] = WANDB_PROJECT
        wandb.login(key=api_key)
        WANDB = '--wandb'
        print(f"W&B is ON: logging to {WANDB_ENTITY}/{WANDB_PROJECT}")
    else:
        WANDB = ''
        print("W&B is OFF (WANDB_API_KEY not found in .env)")
except Exception as e:
    WANDB = ''
    print("W&B is OFF:", e)

## 2 · Verify AVE Dataset Metadata
Checks that the 4,097 AVE videos and 4,143 official annotations exist locally.

In [ ]:
!python -u -m vision.data_ave

## 3 · Frame Extraction (1 fps → 10 frames per clip)
Extracts 10 frames per video (224×224 RGB) to `_local/work/frames_1fps/`.

> **Note:** This skips videos that were already extracted, so you can safely rerun or let it finish in the background.

In [ ]:
!python -u -m vision.extract_frames

## 4 · Build Feature Labels & Metadata
Generates target arrays (`train_y.npy`, `val_y.npy`, `test_y.npy`, `_ids.csv`) in `_local/drive/cache/vision_features_v1/`.

In [ ]:
!python -u -m vision.features

## 5 · Fast Sanity Check (16 Clips Overfit)
Verifies that the model trains, loss drops, and checkpointing works in ~1–2 minutes.

In [ ]:
!python -u -m vision.train --run-id sanity_local --backbone cnn --overfit 16 --epochs 10 --batch 4 --workers 0

## 6 · Local Vision Training (CPU / iGPU Optimized)

For local laptops/desktops, we use **MobileNet-V3 Small**:
- **~1.5M parameters** (vs ~11.7M in ResNet-18).
- Highly optimized depthwise separable convolutions for fast CPU and iGPU forward/backward passes.
- Aligned with team contract: outputs standard **512-D per-second embeddings**.
- `--workers 0`: eliminates Windows multiprocessing process-spawning bottlenecks.

In [ ]:
RUN_ID = 'mobilenet_10ep'
BACKBONE = 'mobilenet_v3_small'  # choices: 'mobilenet_v3_small', 'cnn', 'resnet18'
# BACKBONE = 'resnet18'  # choices: 'mobilenet_v3_small', 'cnn', 'resnet18'
EPOCHS = 10                     # 10 epochs for practical local run; set to 30 for full schedule
BATCH = 8                       # 8 is optimal for CPU/iGPU memory

!python -u -m vision.train --run-id {RUN_ID} --backbone {BACKBONE} --epochs {EPOCHS} --batch {BATCH} --lr 5e-4 --balance sqrt --workers 0 {WANDB}

## 7 · Test Set Evaluation & Confusion Matrix
Evaluates on the official AVE test split (402 clips) and outputs `results.json` and `confusion_test.png`.

In [ ]:
!python -u -m vision.evaluate --run-id {RUN_ID} --batch {BATCH} --workers 0 {WANDB}

import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = Path(f'_local/drive/artifacts/vision_baseline/{RUN_ID}')
results_file = run_dir / 'results.json'
if results_file.exists():
    print("\n--- EVALUATION METRICS ---")
    print(results_file.read_text())

confusion_file = run_dir / 'confusion_test.png'
if confusion_file.exists():
    plt.figure(figsize=(10, 9))
    img = mpimg.imread(str(confusion_file))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Test Set Confusion Matrix', fontsize=14)
    plt.show()

## 8 · Export Embeddings for Multimodal Fusion
Exports 10-second per-clip visual embeddings `(N, 10, 512)` float16 to:
`_local/drive/embeddings/mobilenet_10ep/{split}_emb.npy`.

These embeddings align 1-to-1 with `audio/`'s exported embeddings for late fusion (`512-D + 256-D = 768-D`).

In [ ]:
!python -u -m vision.export_embeddings --run-id {RUN_ID} --batch {BATCH} --workers 0